# Proof of Concept Experiment

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotnine as pn
from sklearn.manifold import smacof

from dataset_similarity.constants import PROJECT_DIR

In [ ]:
# Directory for plot results
plots_dir = PROJECT_DIR / "plots"
plots_dir.mkdir(exist_ok=True)

In [ ]:
# Directory containing metric results
metrics_dir = PROJECT_DIR / "results" / "metrics"


# Fn for reading json into dict
def read_json(json_path: Path) -> dict:
    with open(json_path) as f:
        return json.load(f)


# Read all metric results into a list of dicts and create a DataFrame
results = pd.DataFrame(
    [read_json(json_path) for json_path in metrics_dir.glob("experiment_0_*.json")]
)
results

In [ ]:
# List of datasets in the results
datasets = sorted(set(results["dataset1"].unique()) | set(results["dataset2"].unique()))

In [ ]:
# Fn for populating a distance matrix for a given metric
def populate_distance_matrix(
    datasets: list[str], df: pd.DataFrame, metric: str
) -> pd.DataFrame:
    distance_matrix = pd.DataFrame(0.0, index=datasets, columns=datasets, dtype=float)
    for _, row in df.iterrows():
        d1, d2 = row["dataset1"], row["dataset2"]
        distance_matrix.loc[d1, d2] = row[metric]
        distance_matrix.loc[d2, d1] = row[metric]
    return distance_matrix

In [ ]:
# Fn for produceing MDS solutions for a given distance matrix
def compute_mds_solutions(distance_matrix, max_dims=5):
    return [
        smacof(
            dissimilarities=distance_matrix.to_numpy(),
            metric=True,
            n_components=i,
        )
        for i in range(1, max_dims + 1)
    ]

In [ ]:
# Fn for producing a scree plot of MDS stress vs. number of components
def plot_mds_stress(solutions):
    stress_df = pd.DataFrame(
        {
            "n_components": range(1, len(solutions) + 1),
            "stress": [solution[1] for solution in solutions],
        }
    )
    return (
        pn.ggplot(stress_df, pn.aes(x="n_components", y="stress"))
        + pn.geom_line()
        + pn.geom_point()
        + pn.scale_x_continuous(breaks=range(1, len(solutions) + 1))
        + pn.labs(
            title="MDS Stress vs. Number of Components",
            x="Number of Components",
            y="Stress",
        )
        + pn.theme_bw()
        + pn.theme(aspect_ratio=1)
    )

## OTDD

In [ ]:
distance_matrix_otdd = populate_distance_matrix(datasets, results, "otdd_exact")
distance_matrix_otdd

In [ ]:
solutions_otdd = compute_mds_solutions(distance_matrix_otdd, max_dims=5)

In [ ]:
p = plot_mds_stress(solutions_otdd)
p.save(plots_dir / "poc_otdd_stress.png")
p

In [ ]:
name_to_alpha_map = {
    "domainnet_real_1000": 1.0,
    "domainnet_clipart_1000": 0.0,
    "experiment_0_alpha_0.25": 0.25,
    "experiment_0_alpha_0.5": 0.5,
    "experiment_0_alpha_0.75": 0.75,
}

solution_otdd = solutions_otdd[0][0]  # 1D solution
solution_df_otdd = (
    pd.DataFrame(solution_otdd, columns=["MDS_Dim1"], index=datasets)
    .reset_index()
    .rename(columns={"index": "dataset"})
)
solution_df_otdd["alpha"] = solution_df_otdd.apply(
    lambda row: name_to_alpha_map[row["dataset"]], axis=1
)

p = (
    pn.ggplot(solution_df_otdd, pn.aes(y="MDS_Dim1", x="alpha", label="dataset"))
    + pn.geom_point()
    + pn.geom_text(adjust_text={"arrowprops": {"arrowstyle": "->"}})
    + pn.theme_bw()
    + pn.theme(aspect_ratio=1)
)
p.save(plots_dir / "poc_otdd_smacof_1d.png")
p

In [ ]:
solution_otdd = solutions_otdd[1][0]  # 2D solution
solution_df_otdd = (
    pd.DataFrame(solution_otdd, columns=["MDS_Dim1", "MDS_Dim2"], index=datasets)
    .reset_index()
    .rename(columns={"index": "dataset"})
)

p = (
    pn.ggplot(solution_df_otdd, pn.aes(x="MDS_Dim1", y="MDS_Dim2", label="dataset"))
    + pn.geom_point()
    + pn.geom_text(adjust_text={"arrowprops": {"arrowstyle": "->"}})
    + pn.coord_cartesian(xlim=(-300, 300), ylim=(-300, 300))
    + pn.theme_bw()
    + pn.theme(aspect_ratio=1)
)
p.save(plots_dir / "poc_otdd_smacof_2d.png")
p

## MMD

In [ ]:
distance_matrix_mmd = populate_distance_matrix(datasets, results, "mmd")
distance_matrix_mmd

In [ ]:
solutions_mmd = compute_mds_solutions(distance_matrix_mmd, max_dims=5)

In [ ]:
p = plot_mds_stress(solutions_mmd)
p.save(plots_dir / "poc_mmd_stress.png")
p

In [ ]:
name_to_alpha_map = {
    "domainnet_real_1000": 1.0,
    "domainnet_clipart_1000": 0.0,
    "experiment_0_alpha_0.25": 0.25,
    "experiment_0_alpha_0.5": 0.5,
    "experiment_0_alpha_0.75": 0.75,
}

solution_mmd = solutions_mmd[0][0]  # 1D solution
solution_df_mmd = (
    pd.DataFrame(solution_mmd, columns=["MDS_Dim1"], index=datasets)
    .reset_index()
    .rename(columns={"index": "dataset"})
)
solution_df_mmd["alpha"] = solution_df_mmd.apply(
    lambda row: name_to_alpha_map[row["dataset"]], axis=1
)

p = (
    pn.ggplot(solution_df_mmd, pn.aes(y="MDS_Dim1", x="alpha", label="dataset"))
    + pn.geom_point()
    + pn.geom_text(adjust_text={"arrowprops": {"arrowstyle": "->"}})
    + pn.theme_bw()
    + pn.theme(aspect_ratio=1)
)
p.save(plots_dir / "poc_mmd_smacof_1d.png")
p

In [ ]:
solutions_mmd = solutions_mmd[1][0]  # 2D solution
solution_df_mmd = (
    pd.DataFrame(solutions_mmd, columns=["MDS_Dim1", "MDS_Dim2"], index=datasets)
    .reset_index()
    .rename(columns={"index": "dataset"})
)

p = (
    pn.ggplot(solution_df_mmd, pn.aes(x="MDS_Dim1", y="MDS_Dim2", label="dataset"))
    + pn.geom_point()
    + pn.geom_text(adjust_text={"arrowprops": {"arrowstyle": "->"}})
    + pn.theme_bw()
    + pn.theme(aspect_ratio=1)
)
p.save(plots_dir / "poc_mmd_smacof_2d.png")
p

In [ ]:
results_1 = pd.DataFrame(
    [read_json(json_path) for json_path in metrics_dir.glob("experiment_1_*.json")]
)
results_1

## OT Sinkhorn

In [ ]:
distance_matrix_ot_sinkhorn = populate_distance_matrix(
    datasets, results_1, "ot_sinkhorn"
)
distance_matrix_ot_sinkhorn

In [ ]:
solutions_ot_sinkhorn = compute_mds_solutions(distance_matrix_ot_sinkhorn, max_dims=5)

In [ ]:
p = plot_mds_stress(solutions_ot_sinkhorn)
p.save(plots_dir / "poc_ot_sinkhorn_stress.png")
p

In [ ]:
solution_ot_sinkhorn = solutions_ot_sinkhorn[0][0]  # 1D solution
solution_df_ot_sinkhorn = (
    pd.DataFrame(solution_ot_sinkhorn, columns=["MDS_Dim1"], index=datasets)
    .reset_index()
    .rename(columns={"index": "dataset"})
)
solution_df_ot_sinkhorn["alpha"] = solution_df_ot_sinkhorn["dataset"].map(
    name_to_alpha_map
)

p = (
    pn.ggplot(solution_df_ot_sinkhorn, pn.aes(y="MDS_Dim1", x="alpha", label="dataset"))
    + pn.geom_point()
    + pn.geom_text(adjust_text={"arrowprops": {"arrowstyle": "->"}})
    + pn.theme_bw()
    + pn.theme(aspect_ratio=1)
)
p.save(plots_dir / "poc_ot_sinkhorn_smacof_1d.png")
p

In [ ]:
solution_ot_sinkhorn_2d = solutions_ot_sinkhorn[1][0]  # 2D solution
solution_df_ot_sinkhorn_2d = (
    pd.DataFrame(
        solution_ot_sinkhorn_2d, columns=["MDS_Dim1", "MDS_Dim2"], index=datasets
    )
    .reset_index()
    .rename(columns={"index": "dataset"})
)

p = (
    pn.ggplot(
        solution_df_ot_sinkhorn_2d, pn.aes(x="MDS_Dim1", y="MDS_Dim2", label="dataset")
    )
    + pn.geom_point()
    + pn.geom_text(adjust_text={"arrowprops": {"arrowstyle": "->"}})
    + pn.theme_bw()
    + pn.theme(aspect_ratio=1)
)
p.save(plots_dir / "poc_ot_sinkhorn_smacof_2d.png")
p

## OT Exact

In [ ]:
distance_matrix_ot_exact = populate_distance_matrix(datasets, results_1, "ot_exact")
distance_matrix_ot_exact

In [ ]:
solutions_ot_exact = compute_mds_solutions(distance_matrix_ot_exact, max_dims=5)

In [ ]:
p = plot_mds_stress(solutions_ot_exact)
p.save(plots_dir / "poc_ot_exact_stress.png")
p

In [ ]:
solution_ot_exact = solutions_ot_exact[0][0]  # 1D solution
solution_df_ot_exact = (
    pd.DataFrame(solution_ot_exact, columns=["MDS_Dim1"], index=datasets)
    .reset_index()
    .rename(columns={"index": "dataset"})
)
solution_df_ot_exact["alpha"] = solution_df_ot_exact["dataset"].map(name_to_alpha_map)

p = (
    pn.ggplot(solution_df_ot_exact, pn.aes(y="MDS_Dim1", x="alpha", label="dataset"))
    + pn.geom_point()
    + pn.geom_text(adjust_text={"arrowprops": {"arrowstyle": "->"}})
    + pn.theme_bw()
    + pn.theme(aspect_ratio=1)
)
p.save(plots_dir / "poc_ot_exact_smacof_1d.png")
p

In [ ]:
solution_ot_exact_2d = solutions_ot_exact[1][0]  # 2D solution
solution_df_ot_exact_2d = (
    pd.DataFrame(solution_ot_exact_2d, columns=["MDS_Dim1", "MDS_Dim2"], index=datasets)
    .reset_index()
    .rename(columns={"index": "dataset"})
)

p = (
    pn.ggplot(
        solution_df_ot_exact_2d, pn.aes(x="MDS_Dim1", y="MDS_Dim2", label="dataset")
    )
    + pn.geom_point()
    + pn.geom_text(adjust_text={"arrowprops": {"arrowstyle": "->"}})
    + pn.theme_bw()
    + pn.theme(aspect_ratio=1)
)
p.save(plots_dir / "poc_ot_exact_smacof_2d.png")
p

## OTCE

In [ ]:
distance_matrix_otce = populate_distance_matrix(datasets, results_1, "otce_sinkhorn")
distance_matrix_otce = (
    -distance_matrix_otce
)  # otce is negative; negate for use as distance
distance_matrix_otce

In [ ]:
solutions_otce = compute_mds_solutions(distance_matrix_otce, max_dims=5)

In [ ]:
p = plot_mds_stress(solutions_otce)
p.save(plots_dir / "poc_otce_stress.png")
p

In [ ]:
solution_otce = solutions_otce[0][0]  # 1D solution
solution_df_otce = (
    pd.DataFrame(solution_otce, columns=["MDS_Dim1"], index=datasets)
    .reset_index()
    .rename(columns={"index": "dataset"})
)
solution_df_otce["alpha"] = solution_df_otce["dataset"].map(name_to_alpha_map)

p = (
    pn.ggplot(solution_df_otce, pn.aes(y="MDS_Dim1", x="alpha", label="dataset"))
    + pn.geom_point()
    + pn.geom_text(adjust_text={"arrowprops": {"arrowstyle": "->"}})
    + pn.theme_bw()
    + pn.theme(aspect_ratio=1)
)
p.save(plots_dir / "poc_otce_smacof_1d.png")
p

In [ ]:
solution_otce_2d = solutions_otce[1][0]  # 2D solution
solution_df_otce_2d = (
    pd.DataFrame(solution_otce_2d, columns=["MDS_Dim1", "MDS_Dim2"], index=datasets)
    .reset_index()
    .rename(columns={"index": "dataset"})
)

p = (
    pn.ggplot(solution_df_otce_2d, pn.aes(x="MDS_Dim1", y="MDS_Dim2", label="dataset"))
    + pn.geom_point()
    + pn.geom_text(adjust_text={"arrowprops": {"arrowstyle": "->"}})
    + pn.theme_bw()
    + pn.theme(aspect_ratio=1)
)
p.save(plots_dir / "poc_otce_smacof_2d.png")
p